# 추정
### Tips 데이터셋을 사용하여 모평균, 모평균 차, 모비율 차의 추정 및 표본의 크기 결정, 문제를 실습

– Tips 데이터셋: Seaborn 라이브러리 내장 (1990년대 초반, 미국의 한 레스토랑 웨이터가 약
두 달 반 동안 자신이 받은 팁을 기록한 실제 데이터)

 데이터 분석 포인트

– 식당 손님들은 한 테이블당 평균적으로 얼마를 결제할까?

→ 모평균의 추정

• 전체 손님의 평균 결제액에 대한 95% 신뢰구간 계산

– 점심 손님과 저녁 손님이 내는 팁 금액의 평균에 차이가 있을까?

→ 모평균 차의 추정

• 시간대별 평균 팁 금액 차이의 95% 신뢰구간 계산

– 남성과 여성 고객 중 누가 더 흡연 구역을 많이 찾을까?

→ 모비율 차의 추정

• 성별에 따른 흡연자 비율 차이의 95% 신뢰구간 계산

– 결제액 평균을 오차 $1 이내로 정확히 추정하려면 몇 명을 더 조사해야 할까?

→ 표본의 크기 결정

• 현재 데이터의 표준편차를 기반으로 필요 최소 표본 수를 계산

In [19]:
import seaborn as sns
import numpy as np
from scipy import stats

# Tips 데이터셋 로드 (Seaborn 내장)
tips = sns.load_dataset('tips')

# [1] 모평균 추정: 테이블당 평균 결제액의 95% 신뢰구간
bill = tips['total_bill']
n = len(bill)
mean = bill.mean()
se = bill.std(ddof=1) / np.sqrt(n) # 표준오차
ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
print(f"[1. 모평균 추정] 테이블당 평균 결제액")
print(f" -> 표본 평균: ${mean:.2f}")
print(f" -> 95% 신뢰구간: ${ci[0]:.2f} ~ ${ci[1]:.2f}")


# [2] 모평균 차 추정: 점심 팁 평균 - 저녁 팁 평균
lunch = tips[tips['time']=='Lunch']['tip']
dinner = tips[tips['time']=='Dinner']['tip']
m1, m2 = lunch.mean(), dinner.mean()
v1, v2 = lunch.var(ddof=1), dinner.var(ddof=1)
n1, n2 = len(lunch), len(dinner)

# 두 집단의 분산/표본 수가 다를 때 자유도(Welch) 보정
df = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
se_diff = np.sqrt(v1/n1 + v2/n2)
ci_diff = stats.t.interval(0.95, df=df, loc=m1-m2, scale=se_diff)
print(f"[2. 모평균 차의 추정] 점심 평균 팁 - 저녁 평균 팁")
print(f" -> 평균 팁 차이: ${m1-m2:.2f}")
print(f" -> 95% 신뢰구간: ${ci_diff[0]:.2f} ~ ${ci_diff[1]:.2f}")

# [3] 모비율 차 추정: 남성 흡연율 - 여성 흡연율
male = tips[tips['sex']=='Male']
female = tips[tips['sex']=='Female']
p1 = (male['smoker'] =='Yes').mean()
p2 = (female['smoker']=='Yes').mean()
n1, n2 = len(male), len(female)
se_p = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
z = stats.norm.ppf(0.975) # 95% 양측 임계값
lo, hi = (p1-p2) - z*se_p, (p1-p2) + z*se_p
print(f"[3. 모비율 차의 추정] 남성 흡연율 - 여성 흡연율")
print(f" -> 흡연율 차이: {(p1-p2)*100:.1f}%p")
print(f" -> 95% 신뢰구간: {lo*100:.1f}%p ~ {hi*100:.1f}%p")

# [4] 표본 크기 결정: 결제액 평균 오차 $1 이내
sigma = bill.std(ddof=1) #모표준편차 추정
d = 1.0
n_required = int(np.ceil((z*sigma/d)**2))
print(f"[4. 표본의 크기 결정] 오차 $1 이내")
print(f" -> 현재 표본 크기: {len(bill)}명")
print(f" -> 필요한 최소 표본 크기: {n_required}명")

[1. 모평균 추정] 테이블당 평균 결제액
 -> 표본 평균: $19.79
 -> 95% 신뢰구간: $18.66 ~ $20.91
[2. 모평균 차의 추정] 점심 평균 팁 - 저녁 평균 팁
 -> 평균 팁 차이: $-0.37
 -> 95% 신뢰구간: $-0.73 ~ $-0.02
[3. 모비율 차의 추정] 남성 흡연율 - 여성 흡연율
 -> 흡연율 차이: 0.3%p
 -> 95% 신뢰구간: -12.4%p ~ 13.0%p
[4. 표본의 크기 결정] 오차 $1 이내
 -> 현재 표본 크기: 244명
 -> 필요한 최소 표본 크기: 305명


####모평균 추정 (테이블당 평균 결제액)
- 표본 평균은 $19.79이고 95% 신뢰구간은 $18.66 ~ $20.91이다.

- 즉, 전체 손님 한 테이블당 평균 결제액이 약 $18.66 ~ $20.91 사이에 존재한다고 95%
신뢰수준에서 추정할 수 있다.

####모평균 차의 추정 (점심 팁 - 저녁 팁)

- 평균 팁 차이는 -$0.37이며 95% 신뢰구간은 [-$0.73,
-$0.02]이다.

- 신뢰구간이 0을 포함하지 않으므로 저녁 손님이 점심 손님보다 통계적으로 유의미하게 더 많은
팁을 낸다고 볼 수 있다.
모비율 차의 추정 (남성 흡연율 - 여성 흡연율)

- 흡연율 차이는 약 0.3%p, 95% 신뢰구간은 [-12.4%p, 13.0%p]이다.

- 신뢰구간이 0을 포함하므로 성별에 따른 흡연 구역 이용률 차이는 통계적으로 유의하지 않다.

####표본의 크기 결정 (오차 $1 이내 추정)

- 현재 표본 크기는 244명이며, 표본 표준편차를 기반으로 계산한 필요 최소 표본 크기는 305명이다.

- 따라서 약 61명을 추가로 조사해야 결제액 평균을 오차 $1 이내로 정확히 추정할 수 있다.